# Dubai Review Intelligence: bilingual (Arabic/English) review analysis
MAAI 902 - NLP & LLMs | Individual project

Pipeline: pre-processing -> sentiment (baseline vs zero-shot vs fine-tuned XLM-R) -> aspect sentiment -> complaint clustering -> grounded LLM briefing + faithfulness check.

**Before running:** Runtime > Change runtime type > **T4 GPU**. Run cells top to bottom.

## 1. Setup

In [ ]:
!pip -q install transformers sentence-transformers gradio faiss-cpu anthropic
import os, sys, zipfile, glob
# Upload dubai-review-intelligence.zip using the Files panel (left), then:
if os.path.exists('dubai-review-intelligence.zip'):
    zipfile.ZipFile('dubai-review-intelligence.zip').extractall('.')
if os.path.isdir('dubai-review-intelligence'):
    os.chdir('dubai-review-intelligence')
sys.path.insert(0, os.getcwd())
print(os.getcwd(), os.listdir('.'))

## 2. Data
Upload your two CSV files into `data/` (see README, Step "Get the data"):
* Arabic reviews CSV  -> set `AR_FILE`, text column, label column
* English reviews CSV -> set `EN_FILE`, text column, label column

`star_scale=True` means the label column holds 1-5 stars; for Positive/Negative/Mixed words it is detected automatically.

In [ ]:
import pandas as pd, numpy as np
from src import data, preprocess

AR_FILE, AR_TEXT, AR_LABEL = 'data/ar_reviews_100k.tsv', 'text', 'label'
EN_FILE, EN_TEXT, EN_LABEL = 'data/english_reviews.csv', 'text', 'label'

ar = data.load_csv(AR_FILE, AR_TEXT, AR_LABEL, source='arabic')
en = data.load_csv(EN_FILE, EN_TEXT, EN_LABEL, source='english')
print('Arabic before domain filter:', len(ar))
ar = data.filter_hospitality(ar)   # keep hotel/restaurant reviews only (dataset also has books)
print('Arabic hospitality reviews:', len(ar), '| English:', len(en))
print(ar.label.value_counts())
ar = data.balance_by_label(ar, 3000)       # <= 9k Arabic
en = data.balance_by_label(en, 3000)       # <= 9k English
cs = data.make_code_switched(ar, en, n=3000)   # synthetic Arabic+English reviews
full = pd.concat([ar, en, cs], ignore_index=True)
full = preprocess.preprocess_dataframe(full)
print(full.groupby(['source','label']).size().unstack())
print(full.lang.value_counts())

In [ ]:
train, val, test = data.stratified_split(full)
print(len(train), len(val), len(test))
# EDA figures for the report
import matplotlib.pyplot as plt, os
os.makedirs('results', exist_ok=True)
fig, ax = plt.subplots(1, 2, figsize=(9, 3.2))
full.label.map({0:'neg',1:'neu',2:'pos'}).value_counts().plot.bar(ax=ax[0], title='Label distribution')
full.lang.value_counts().plot.bar(ax=ax[1], title='Language mix')
fig.tight_layout(); fig.savefig('results/eda.png', dpi=150)
full['n_words'] = full.clean_text.str.split().str.len()
print(full.n_words.describe())

## 3. Models: baseline, zero-shot, fine-tuned

In [ ]:
from src import train as T, evaluate as E, models
base = T.train_tfidf_baseline(train)
p_base = base.predict(test.clean_text_agg)
results = [E.evaluate_predictions(test, p_base, 'TF-IDF + LogReg')]
print(results[-1])

In [ ]:
zs = models.HFSentiment(models.ZERO_SHOT_MODEL)
p_zs = zs.predict_labels(test.clean_text.tolist())
results.append(E.evaluate_predictions(test, p_zs, 'XLM-R sentiment (zero-shot)'))
print(results[-1])

In [ ]:
# Fine-tune XLM-R base (about 10-25 min on a T4 for ~20k reviews)
hist = T.finetune(train, val, base_model='xlm-roberta-base', out_dir='outputs/xlmr-sentiment',
                  epochs=3, lr=2e-5, batch_size=32, max_len=128)
pd.DataFrame(hist)

In [ ]:
ft = models.HFSentiment('outputs/xlmr-sentiment')
p_ft = ft.predict_labels(test.clean_text.tolist())
results.append(E.evaluate_predictions(test, p_ft, 'XLM-R fine-tuned (ours)'))
table = E.comparison_table(results); table

In [ ]:
table.to_csv('results/model_comparison.csv')
E.plot_confusion(test.label, p_ft, 'Fine-tuned XLM-R', 'results/confusion_finetuned.png')
errs = E.error_analysis(test, p_ft, n=40); errs.to_csv('results/errors.csv', index=False)
errs.head(10)

## 4. Full pipeline on example reviews
Uses the fine-tuned model for sentiment, a multilingual sentence-transformer for clustering, and an LLM briefing.
Set `ANTHROPIC_API_KEY` to use Claude; otherwise the template backend is used. `backend='local'` uses an open Hugging Face model.

In [ ]:
from src import pipeline, cluster
sample = pd.read_csv('data/sample_reviews.csv')
embed_fn = cluster.default_embed_fn()
res = pipeline.run_pipeline(sample, ft, embed_fn, backend='auto')
display(res['aspect_summary'])
for c in res['clusters']: print(c['size'], c['top_aspect'], c['terms'])
print(res['summary']); print(res['verification'])

In [ ]:
# Faithfulness over the LLM backend(s) you can run; report the numbers.
from src.summarize import get_backend, verify_summary
rows = []
for name in ['template', 'auto', 'local']:
    try:
        bk = get_backend(name); txt = bk.generate(res['evidence'])
        v = verify_summary(txt, res['evidence']); rows.append({'backend': bk.name, **v})
    except Exception as e:
        print(name, 'skipped:', e)
pd.DataFrame(rows)

In [ ]:
# Performance on real-world-style mixed text: evaluate on the code-switched subset only
mixed = test[test.lang == 'mixed']
print(len(mixed), 'mixed reviews in test set')
print('F1 mixed (fine-tuned):', E.f1_score(mixed.label, np.array(p_ft)[(test.lang=='mixed').values], average='macro') if len(mixed) else 'n/a')

## 5. Launch the demo (public link appears below)

In [ ]:
!python app.py